# Financial Data Extraction

> Module 1 - Markets, Data, and EDA

Introduce programmatic data extraction for market prices and macroeconomic series using Python data providers.

## Learning objectives
- Download market data for multiple assets using `yfinance`.
- Inspect the structure returned by a market data API.
- Retrieve macroeconomic series through FRED as a reference external source.
- Identify common data-provider risks such as missing values, changing schemas, and network dependency.

## Lesson flow
1. Set up imports and dates.
2. Download and inspect stock data.
3. Retrieve benchmark index data.
4. Query macroeconomic data and close with source references.

## Student deliverable
A short data inventory describing each downloaded source, frequency, date range, and main limitations.


## Setup

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# import pandas_datareader.data as web
import yfinance as yf


pd.set_option("display.max_columns",80)

In [ ]:
yf.__version__

## Stock Data

In [ ]:
yesterday = str(date.today() - timedelta(days = 1))
print("Today's date:", yesterday)

In [ ]:
start_date = "2019-01-01"
tickers = ['MAT','DIS','KO', 'NVDA','PFE','MRNA', "AAPL", "META", "TSLA", "^GSPC"]
all_data = yf.download(tickers, start_date, yesterday)

In [ ]:
all_data["Adj Close"]["AAPL"]

In [ ]:
np.max(all_data["Adj Close"]["AAPL"])

In [ ]:
type(all_data)

In [ ]:
all_data.info()

In [ ]:
all_data.head(2)

In [ ]:
(
    all_data.Volume
).drop(columns='^GSPC').plot(
    figsize=(12,8)
);

In [ ]:
all_data.Volume.drop(columns='^GSPC').groupby(
    pd.PeriodIndex(
        all_data.index, 
        freq="M"
    )
).sum().plot(
    figsize=(12,8)
);

In [ ]:
(
    (all_data.Open - all_data["Adj Close"]) / all_data.Open
).plot(
    figsize=(12,8)
);

In [ ]:
(
    (all_data.Open - all_data["Adj Close"]) / all_data.Open
).groupby(
    pd.PeriodIndex(
        all_data.index, 
        freq="M"
    )
).mean().plot(
    figsize=(12,8)
);

In [ ]:
price_data = all_data["Adj Close"]
price_data.head()

In [ ]:
price_data.info()

### S&P 500 

In [ ]:
tickers_sp500 = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]
tickers_sp500.head()

In [ ]:
tickers_sp500.Founded.value_counts().sort_values().tail(25).sort_index().plot(
    kind='barh',
    figsize=(8,7)
);

In [ ]:
tickers_sp500[
    tickers_sp500.Founded == "1886"
]

In [ ]:
tickers_sp500[
    tickers_sp500.Founded.min() == tickers_sp500.Founded
]

In [ ]:
sp_price_data = yf.download(tickers_sp500.Symbol.to_list(),'2022-12-01','2022-12-31', auto_adjust=True)['Close']
sp_price_data.head()

In [ ]:
sp_price_data.mean().sort_values().dropna().tail(10)

## fredapi

** Agregar ss para crear api key en freapi

In [ ]:
import pandas as pd 
import requests 
import json 
# import plotly.graph_objects as go
from fredapi import Fred

In [ ]:
from api_keys import fred_api_key
fred = Fred(api_key=fred_api_key)
sp500_index_data = fred.get_series('SP500')

In [ ]:
sp500_index_data.plot();

In [ ]:
gdp_data = fred.get_series_first_release('GDP')
sp500_index_data.plot();

In [ ]:
fred.search('fed rate').T

In [ ]:
personal_income_series = fred.search_by_release(175, limit=5, order_by='popularity', sort_order='desc')

In [ ]:
personal_income_series['title']

In [ ]:
df = {}
df['SF'] = fred.get_series('PCPI06075')
df['NY'] = fred.get_series('PCPI36061')
df['DC'] = fred.get_series('PCPI11001')
df = pd.DataFrame(df)
df.plot()

In [ ]:
df = fred.search_by_category(101, limit=10, order_by='popularity', sort_order='desc')
df['title']

In [ ]:
state_df = df[~df['title'].str.startswith('Per Capita Personal Income in the')]

In [ ]:
len(state_df)

In [ ]:
state_df.id.str[:2]

In [ ]:
fred.search("potential inflation").T

In [ ]:
fred.search("Mexico").T

In [ ]:
fred.search("CPIEALL").T

In [ ]:
inf=fred.get_series("CPIEALL")
# inf=inf["2009–12–01":"2023–01–01"]

In [ ]:
inf.plot();

In [ ]:
inf_quarterly=inf.resample("Q").mean()
inf_growth=inf_quarterly.pct_change().dropna()
inf_growth.plot(figsize=(10,7));

In [ ]:
inf_growth.tail(24).iloc[:-1].plot();

## References
- https://technically.substack.com/p/whats-an-api
- https://technically.dev/posts/apis-for-the-rest-of-us
- https://aroussi.com/post/python-yahoo-finance
- https://github.com/ranaroussi/yfinance
- https://fred.stlouisfed.org